In [1]:
import pandas as pd


In [2]:
def load_c_elegans_sequence_into_dictionaries():
    # loading c_elegans sequences
    c_elegans_gpcr = pd.read_csv("Data/C. Elegans GPCR.csv")
    gpcr_seqs = dict(zip(
        c_elegans_gpcr["GPCR name"],
        c_elegans_gpcr["GPCR sequence"],
    ))

    c_elegans_peptides = pd.read_csv("Data/C. Elegans Peptides.csv")
    peptide_seqs = dict(zip(
        c_elegans_peptides["Peptide name"],
        c_elegans_peptides["Peptide sequence"],
    ))
    return gpcr_seqs, peptide_seqs

gpcr_seqs, peptide_seqs = load_c_elegans_sequence_into_dictionaries()

#loading labels
c_elegans_pairs = pd.read_csv("Data/C_Elegans_Agonist_Labels.csv")
#family is simply defined by removing the last two chars from the GPCR name
#For example, NPR-8-1 and NPR-8-2  both belong to the family NPR-8
c_elegans_pairs["gpcr_family"] = c_elegans_pairs["GPCR name"].str[:-2]

In [3]:
#!!! load human gpcrs and agonist pairs, along with platynereis
#these are for testing on generalization to other species, they are in the data folder but not set up

In [4]:
#phylogenetic splits (gpcr)
# each group in the list is a phylogenetically similar group of GPCRs that should not be split into different folds
TEST_GPCR_FAMILY_GROUPS = [['FRPR-16', 'FRPR-18', 'FRPR-4', 'FRPR-6', 'NPR-22', 'NMUR-2'],
                           ['AEX-2', 'DMSR-5', 'DMSR-6', 'DMSR-7', 'DMSR-8','NPR-32'],
                           ['FRPR-7', 'FRPR-7', 'NPR-6', 'GNRR-3', 'EGL-6'],
                           ['TKR-1', 'TKR-2', 'DMSR-1', 'DMSR-2', 'NPR-40', 'GNRR-6'],
                           ['NPR-42', 'FRPR-9', 'FRPR-15', 'FRPR-19', 'NMUR-1'],
                           ['NPR-8', 'NPR-24', 'NPR-37', 'NPR-43', 'FRPR-3'],
                           ['FRPR-8', 'GNRR-1', 'SPRR-1', 'SPRR-2', 'NPR-11', 'NPR-12'],
                           ['NPR-41', 'NPR-1', 'NPR-2', 'NPR-3', 'PDFR-1', 'NTR-1'],
                           ['TRHR-1', 'NPR-35', 'NPR-13', 'NPR-5', 'SEB-3'],
                           ['DMSR-3', 'NPR-10', 'NPR-4', 'NPR-39', 'CKR-1', 'CKR-2']]

#rotates gpcr groups so each test fold gets a different family group for hyperparameter tuning
# Cross-validation by GPCR family (leave-related-receptors-out).
#this uses a cross-validation approach
# instead of one test-train split, the approach holds out a set of closely related receptors
# it does this for each set of similar proteins (validation peptides), and runs training without them
# Because hyperparameter tuning gpcrs is rotated by two, each gpcr group gets a validation set for tuning that is 2 GPCR groups back
def create_folds():
    hyperparameter_tuning_peptides = TEST_GPCR_FAMILY_GROUPS[-2:] + TEST_GPCR_FAMILY_GROUPS[0:-2]
    global train
    folds = []
    for test_families, hpt_families in zip(TEST_GPCR_FAMILY_GROUPS, hyperparameter_tuning_peptides):
        test = c_elegans_pairs[c_elegans_pairs["gpcr_family"].isin(test_families)]
        hyperparameter_tuning = c_elegans_pairs[c_elegans_pairs["gpcr_family"].isin(hpt_families)]  # validation for HPO
        train = c_elegans_pairs[~ c_elegans_pairs["gpcr_family"].isin(test_families)]
        train_hpt = c_elegans_pairs[
            ~ c_elegans_pairs["gpcr_family"].isin(test_families)
            & ~c_elegans_pairs["gpcr_family"].isin(hpt_families)
            ]
        folds.append({
            "test_families": test_families,
            "hpt_families": hpt_families,
            "test": test,
            "hyperparameter_tuning": hyperparameter_tuning,
            "train": train_hpt,
            "train_hpt": train_hpt
        })
        print("test families:", test_families)
        print("validation families", hpt_families)
        print("n test:", len(test), "n train:", len(train), "n hpt:", len(hyperparameter_tuning))
    return folds

folds = create_folds()

test families: ['FRPR-16', 'FRPR-18', 'FRPR-4', 'FRPR-6', 'NPR-22', 'NMUR-2']
validation families ['TRHR-1', 'NPR-35', 'NPR-13', 'NPR-5', 'SEB-3']
n test: 2373 n train: 19662 n hpt: 2034
test families: ['AEX-2', 'DMSR-5', 'DMSR-6', 'DMSR-7', 'DMSR-8', 'NPR-32']
validation families ['DMSR-3', 'NPR-10', 'NPR-4', 'NPR-39', 'CKR-1', 'CKR-2']
n test: 2034 n train: 20001 n hpt: 2034
test families: ['FRPR-7', 'FRPR-7', 'NPR-6', 'GNRR-3', 'EGL-6']
validation families ['FRPR-16', 'FRPR-18', 'FRPR-4', 'FRPR-6', 'NPR-22', 'NMUR-2']
n test: 2034 n train: 20001 n hpt: 2373
test families: ['TKR-1', 'TKR-2', 'DMSR-1', 'DMSR-2', 'NPR-40', 'GNRR-6']
validation families ['AEX-2', 'DMSR-5', 'DMSR-6', 'DMSR-7', 'DMSR-8', 'NPR-32']
n test: 2373 n train: 19662 n hpt: 2034
test families: ['NPR-42', 'FRPR-9', 'FRPR-15', 'FRPR-19', 'NMUR-1']
validation families ['FRPR-7', 'FRPR-7', 'NPR-6', 'GNRR-3', 'EGL-6']
n test: 2373 n train: 19662 n hpt: 2034
test families: ['NPR-8', 'NPR-24', 'NPR-37', 'NPR-43', 'FRPR-3

In [5]:
peptides_needed = sorted(c_elegans_pairs["Peptide"].unique())
gpcrs_needed = sorted(c_elegans_pairs["GPCR name"].unique())

missing_p = [p for p in peptides_needed if p not in peptide_seqs]
missing_g = [g for g in gpcrs_needed if g not in gpcr_seqs]
#assert that all needed peptides are present
assert not missing_p
assert not missing_g

for peptide in peptides_needed:
    peptide_seq = peptide_seqs[peptide]
    #!!!do something with esm2 to get embeddings
    pass

for gpcr in gpcrs_needed:
    gpcr_seq = gpcr_seqs[gpcr]
    #!!! do something with esm2 to get embeddings
    pass

#!!! how is ESM2 fine tuned?

In [8]:
import pickle
from pathlib import Path


def pickle_data():
    """
    Serialize preprocessing artifacts and validate the saved dataset.

    Stores the current GPCR sequences, peptide sequences, labeled interaction
    pairs, cross-validation folds, and held-out GPCR family groups in a pickle
    file. After writing the file, reloads it and verifies that the primary
    collection sizes match the in-memory data used to create the payload. Status
    messages are printed after saving and after successful validation.

    """
    PREPROCESSED_PATH = Path("Data/c_elegans_preprocessed.pkl")

    # Everything later training needs: sequences, labeled pairs, and CV folds.
    preprocessed = {
        "gpcr_seqs": gpcr_seqs,
        "peptide_seqs": peptide_seqs,
        "pairs": c_elegans_pairs,
        "folds": folds,
        "test_gpcr_family_groups": TEST_GPCR_FAMILY_GROUPS,
    }

    with open(PREPROCESSED_PATH, "wb") as f:
        pickle.dump(preprocessed, f)

    print(f"wrote {PREPROCESSED_PATH.resolve()}")

    def load_preprocessed(path=PREPROCESSED_PATH):
        with open(path, "rb") as f:
            return pickle.load(f)

    # reload check
    loaded = load_preprocessed()
    assert len(loaded["gpcr_seqs"]) == len(gpcr_seqs)
    assert len(loaded["peptide_seqs"]) == len(peptide_seqs)
    assert len(loaded["pairs"]) == len(c_elegans_pairs)
    assert len(loaded["folds"]) == len(folds)
    print(
        "loaded ok:",
        f"{len(loaded['gpcr_seqs'])} gpcrs,",
        f"{len(loaded['peptide_seqs'])} peptides,",
        f"{len(loaded['pairs'])} pairs,",
        f"{len(loaded['folds'])} folds",
    )


pickle_data()


wrote C:\Users\natef\PycharmProjects\DL_FinalProject\Data\c_elegans_preprocessed.pkl
loaded ok: 66 gpcrs, 340 peptides, 22035 pairs, 10 folds
